In [20]:
import pandas as pd
import numpy as np
from sklearn import model_selection, linear_model, metrics

In [21]:
ratings = pd.read_table('ml-1m/ratings.dat', delimiter="::", header=None, names=("UserID", "MovieID", "Rating", "Timestamp"))
movies = pd.read_table('ml-1m/movies.dat', delimiter="::", header=None, names = ("MovieID", "Title", "Genres"))

C:\Users\tyane\AppData\Local\Temp\ipykernel_13496\2065777399.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  ratings = pd.read_table('ml-1m/ratings.dat', delimiter="::", header=None, names=("UserID", "MovieID", "Rating", "Timestamp"))
C:\Users\tyane\AppData\Local\Temp\ipykernel_13496\2065777399.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  movies = pd.read_table('ml-1m/movies.dat', delimiter="::", header=None, names = ("MovieID", "Title", "Genres"))


In [46]:
from surprise import KNNWithMeans, KNNBaseline
from surprise.prediction_algorithms.matrix_factorization import SVD, SVDpp
from surprise import Dataset
from surprise import accuracy
from surprise import Reader
from surprise.model_selection.validation import cross_validate

In [23]:
movies_with_ratings = movies.merge(ratings, on='MovieID').reset_index(drop=True)
movies_with_ratings.dropna(inplace=True)
movies_with_ratings

,MovieID,Title,Genres,UserID,Rating,Timestamp
0,1,Toy Story (1995),Animation|Children's|Comedy,1,5,978824268
1,1,Toy Story (1995),Animation|Children's|Comedy,6,4,978237008
2,1,Toy Story (1995),Animation|Children's|Comedy,8,4,978233496
3,1,Toy Story (1995),Animation|Children's|Comedy,9,5,978225952
4,1,Toy Story (1995),Animation|Children's|Comedy,10,5,978226474
...,...,...,...,...,...,...
1000204,3952,"Contender, The (2000)",Drama|Thriller,5812,4,992072099
1000205,3952,"Contender, The (2000)",Drama|Thriller,5831,3,986223125
1000206,3952,"Contender, The (2000)",Drama|Thriller,5837,4,1011902656
1000207,3952,"Contender, The (2000)",Drama|Thriller,5927,1,979852537


## Создадим датасет

In [24]:
dataset = pd.DataFrame({
    'uid': movies_with_ratings.UserID,
    'iid': movies_with_ratings.Title,
    'rating': movies_with_ratings.Rating
})

print("Rating min: {}, max: {}".format(ratings.Rating.min(), ratings.Rating.max()))

Rating min: 1, max: 5


In [ ]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(dataset, reader)
data

In [27]:
dataset['uid'].nunique(), dataset['iid'].nunique()

(6040, 3706)

## KNNWithMeans (k = 50, user based, cosine distance)
### Mean RMSE = 0.936

In [ ]:
knn_50_user_based_cosine = KNNWithMeans(k=50, sim_options={
    'name': 'cosine',
    'user_based': True  # compute  similarities between users
})

cross_validate(algo=knn_50_user_based_cosine, data=data, measures=["RMSE"], cv=5, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9380  0.9345  0.9353  0.9369  0.9360  0.9361  0.0012  
Fit time          33.61   39.24   37.07   35.53   43.83   37.86   3.51    
Test time         72.58   83.74   72.05   79.40   88.22   79.20   6.28    


{'test_rmse': array([0.93795264, 0.93449905, 0.93532715, 0.93693334, 0.93602399]),
 'fit_time': (33.613293170928955,
  39.23939776420593,
  37.06751871109009,
  35.53125023841858,
  43.83105540275574),
 'test_time': (72.57850646972656,
  83.73978638648987,
  72.04710412025452,
  79.40037131309509,
  88.22012948989868)}

## KNNWithMeans (k = 50, item based, cosine distance)
### Mean RMSE = 0.893

In [37]:
knn_50_item_based_cosine = KNNWithMeans(k=50, sim_options={
    'name': 'cosine',
    'user_based': False  # compute  similarities between items
})

cross_validate(algo=knn_50_item_based_cosine, data=data, measures=["RMSE"], cv=5, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8940  0.8934  0.8938  0.8951  0.8885  0.8930  0.0023  
Fit time          17.30   23.17   18.37   22.24   17.42   19.70   2.50    
Test time         47.62   53.89   52.03   43.97   41.96   47.89   4.55    


{'test_rmse': array([0.89397013, 0.89340363, 0.89384256, 0.89512348, 0.88849997]),
 'fit_time': (17.29910945892334,
  23.17074751853943,
  18.368088960647583,
  22.238199710845947,
  17.41976237297058),
 'test_time': (47.61921715736389,
  53.88648223876953,
  52.03396534919739,
  43.96529722213745,
  41.96438479423523)}

## KNNWithMeans (k = 85, item based, cosine distance)
### Mean RMSE = 0.893

In [43]:
knn_80_item_based_cosine = KNNWithMeans(k=80, sim_options={
    'name': 'cosine',
    'user_based': False  # compute  similarities between items
})

cross_validate(algo=knn_80_item_based_cosine, data=data, measures=["RMSE"], cv=5, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8936  0.8929  0.8925  0.8927  0.8959  0.8935  0.0012  
Fit time          15.91   17.26   17.18   17.21   17.03   16.92   0.51    
Test time         44.87   46.32   48.76   46.35   45.82   46.42   1.29    


{'test_rmse': array([0.89355482, 0.89286769, 0.89247676, 0.89268638, 0.89586611]),
 'fit_time': (15.910434246063232,
  17.259755849838257,
  17.178952932357788,
  17.205819606781006,
  17.029430627822876),
 'test_time': (44.87320804595947,
  46.32299327850342,
  48.7631995677948,
  46.34962320327759,
  45.81555199623108)}

## KNNWithMeans (k = 35, item based, cosine distance)
### Mean RMSE = 0.894

In [44]:
knn_35_item_based_cosine = KNNWithMeans(k=35, sim_options={
    'name': 'cosine',
    'user_based': False  # compute  similarities between items
})

cross_validate(algo=knn_35_item_based_cosine, data=data, measures=["RMSE"], cv=5, verbose=True)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8933  0.8934  0.8956  0.8936  0.8956  0.8943  0.0011  
Fit time          16.32   17.09   17.54   17.47   17.57   17.20   0.47    
Test time         38.27   38.82   39.28   40.69   39.09   39.23   0.80    


{'test_rmse': array([0.89329504, 0.89339181, 0.89557245, 0.89358903, 0.89560273]),
 'fit_time': (16.323549270629883,
  17.085269451141357,
  17.53997778892517,
  17.473142862319946,
  17.574856996536255),
 'test_time': (38.27485108375549,
  38.823092460632324,
  39.27749061584473,
  40.69154119491577,
  39.09101438522339)}

## KNNWithMeans (k = 50, item based, pearson distance)
### Mean RMSE = 0.886

In [42]:
knn_50_item_based_pearson = KNNWithMeans(k=50, sim_options={
    'name': 'pearson',
    'user_based': False  # compute  similarities between items
})

cross_validate(algo=knn_50_item_based_pearson, data=data, measures=["RMSE"], cv=5, verbose=True)

Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8852  0.8882  0.8863  0.8871  0.8871  0.8868  0.0010  
Fit time          19.56   19.41   19.27   19.32   20.03   19.52   0.27    
Test time         47.26   41.24   42.97   46.38   42.19   44.01   2.38    


{'test_rmse': array([0.88515481, 0.88822554, 0.88634937, 0.88714453, 0.88710125]),
 'fit_time': (19.556158781051636,
  19.411510467529297,
  19.274982690811157,
  19.317991495132446,
  20.026411533355713),
 'test_time': (47.25936150550842,
  41.242966175079346,
  42.965068101882935,
  46.37881302833557,
  42.19186329841614)}

## KNNBaseline (k = 50, item based, pearson distance)
### Mean RMSE = 0.887

In [47]:
knnb_50_item_based_pearson = KNNBaseline(k=50, sim_options={
    'name': 'pearson',
    'user_based': False  # compute  similarities between items
})

cross_validate(algo=knnb_50_item_based_pearson, data=data, measures=["RMSE"], cv=5, verbose=True)

Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNBaseline on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8870  0.8887  0.8870  0.8863  0.8881  0.8874  0.0009  
Fit time          19.56   20.73   20.57   20.31   20.92   20.42   0.47    
Test time         44.91   44.63   46.13   43.72   44.36   44.75   0.80    


{'test_rmse': array([0.88703372, 0.88870632, 0.88696851, 0.88626646, 0.88808651]),
 'fit_time': (19.562695503234863,
  20.727389812469482,
  20.574495553970337,
  20.30631160736084,
  20.91967248916626),
 'test_time': (44.91498517990112,
  44.63270163536072,
  46.131025314331055,
  43.71609973907471,
  44.359861612319946)}

## SVD default (20 epochs)
### Mean RMSE = 0.873

In [36]:
svd = SVD()

cross_validate(algo=svd, data=data, measures=["RMSE"], cv=5, verbose=True)

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8729  0.8738  0.8734  0.8742  0.8732  0.8735  0.0005  
Fit time          8.40    8.50    8.57    8.99    9.32    8.75    0.35    
Test time         1.30    1.05    1.31    1.36    1.14    1.23    0.12    


{'test_rmse': array([0.87287895, 0.87382096, 0.87341794, 0.87423509, 0.87315689]),
 'fit_time': (8.396216630935669,
  8.502601385116577,
  8.566641807556152,
  8.985471248626709,
  9.316006183624268),
 'test_time': (1.2964413166046143,
  1.0538575649261475,
  1.3100271224975586,
  1.3622238636016846,
  1.1382067203521729)}

## SVD (40 epochs)
### Mean RMSE = 0.894

In [39]:
svd_40 = SVD(n_epochs=40)

cross_validate(algo=svd_40, data=data, measures=["RMSE"], cv=5, verbose=True)

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8947  0.8937  0.8939  0.8948  0.8964  0.8947  0.0010  
Fit time          16.78   16.67   17.81   17.72   17.74   17.34   0.51    
Test time         1.31    1.36    1.32    1.32    1.83    1.43    0.20    


{'test_rmse': array([0.8947379 , 0.89369146, 0.89391647, 0.89481536, 0.89643923]),
 'fit_time': (16.777472019195557,
  16.67318344116211,
  17.80715847015381,
  17.722599267959595,
  17.740114212036133),
 'test_time': (1.3104581832885742,
  1.3641045093536377,
  1.323312759399414,
  1.3235666751861572,
  1.8315520286560059)}

## SVD (15 epochs)
### Mean RMSE = 0.876

In [ ]:
svd_15 = SVD(n_epochs=15)

cross_validate(algo=svd_15, data=data, measures=["RMSE"], cv=5, verbose=True)

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8763  0.8773  0.8779  0.8756  0.8771  0.8768  0.0008  
Fit time          6.51    6.65    6.79    7.00    7.14    6.82    0.23    
Test time         1.54    1.55    1.54    1.59    1.67    1.58    0.05    


{'test_rmse': array([0.87631927, 0.87733433, 0.87785063, 0.87563159, 0.87707608]),
 'fit_time': (6.5120110511779785,
  6.649170875549316,
  6.790823221206665,
  6.99855637550354,
  7.140111207962036),
 'test_time': (1.541752815246582,
  1.5465364456176758,
  1.5401437282562256,
  1.5869174003601074,
  1.6723778247833252)}

## SVDpp (20 epochs)
### Mean RMSE = 0.8620

In [ ]:
svdpp_15 = SVDpp(n_epochs=20)

cross_validate(algo=svdpp_15, data=data, measures=["RMSE"], cv=5, verbose=True)

Evaluating RMSE of algorithm SVDpp on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8617  0.8605  0.8636  0.8621  0.8621  0.8620  0.0010  
Fit time          288.00  446.64  340.55  427.27  315.60  363.61  62.45   
Test time         64.30   88.75   55.71   54.88   169.44  86.62   43.18   


{'test_rmse': array([0.86165497, 0.86051273, 0.86357657, 0.8621082 , 0.86212645]),
 'fit_time': (288.00480246543884,
  446.6414659023285,
  340.5521779060364,
  427.26612401008606,
  315.59850668907166),
 'test_time': (64.30429816246033,
  88.75480937957764,
  55.707948207855225,
  54.879093647003174,
  169.43697834014893)}

## Выводы

- Для алгоритма KNNWithMeans item-based подход дает лучшие результаты, чем user-based.
- Увеличение максимального количества соседей для рассмотрения не уменьшает ошибку
- Изменение косиносного расстояния на критерий Пирсона незначительно уменьшает ошибку при тех же значениях k для KNNWithMeans
- KNNBaseline не показывает ощутимой разницы с KNNWithMeans для тех же значений параметров k и оценки расстояния
- SVD с умолчательными параметрами дает результат лучше чем любая из попыток KNNWithMeans и намного быстрее. Но все равно это больше 8.7
- Единственный вариант, который обеспечил ошибку меньше 8.7 это SVDpp, который на милионном MovieLens работал примерно 37 минут. Зато почитал подробнее про понятие скрытых факторов...